# 13.6 - Human Approval

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

Human-in-the-loop pauses graph execution at a checkpoint so a human can approve, reject, or edit before a sensitive action runs. This keeps agents autonomous but accountable.

## 2. Why Does This Matter?

## 3. Prerequisites

Units 13.1-13.5.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Compile a graph with `interrupt_before` so it pauses
- Resume after human input with `app.invoke(None, config)`
- Implement approve, reject, and modify flows
- Keep an audit log of decisions

## 5. Mental Model

Human approval is a toll booth: the agent drives to the booth, stops, shows what it plans to do, a human waves it through or redirects it — only then does the agent continue. A checkpointer is mandatory for interrupts to work.


## 6. Setup

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


## 7. Propose -> Review (interrupt) -> Execute

The graph runs `propose`, then pauses right before `review`. The human inspects the planned action and decides.

In [2]:
class ApprovalState(TypedDict):
    customer: str
    action: str | None
    approved: bool | None
    result: str | None


def propose(state):
    return {"action": f"Send payment-reminder email to {state['customer']}"}


def review(state):
    return state  # this is a gate; the human decides before it runs


def execute(state):
    if state["approved"]:
        return {"action": state["action"], "approved": True,
                "result": f"Executed: {state['action']}"}
    return {"action": state["action"], "approved": False, "result": "Cancelled by human."}


g = StateGraph(ApprovalState)
g.add_node("propose", propose)
g.add_node("review", review)
g.add_node("execute", execute)
g.add_edge(START, "propose")
g.add_edge("propose", "review")
g.add_edge("review", "execute")
g.add_edge("execute", END)

app = g.compile(checkpointer=MemorySaver(), interrupt_before=["review"])

config = {"configurable": {"thread_id": "approval-1"}}
paused = app.invoke({"customer": "Ada", "action": None, "approved": None, "result": None}, config)
print("PAUSED for human review.")
print("Planned action:", paused["action"], "| approved so far:", paused["approved"])


PAUSED for human review.
Planned action: Send payment-reminder email to Ada | approved so far: None


The human reviews the plan and **approves** (imagine the reviewer clicking "Approve"). Writing the decision with `update_state` and then resuming with `invoke(None, config)` lets the graph continue from the interrupt into `execute`.

In [3]:
app.update_state(config, {"approved": True})
approved = app.invoke(None, config)
print("APPROVED ->", approved["result"])


APPROVED -> Executed: Send payment-reminder email to Ada


A second conversation runs the **reject** path. Note the separate thread ID — approvals never leak across users.

In [4]:
config2 = {"configurable": {"thread_id": "approval-2"}}
app.invoke({"customer": "Bob", "action": None, "approved": None, "result": None}, config2)
app.update_state(config2, {"approved": False})
rejected = app.invoke(None, config2)  # plain resume, no state change
print("REJECTED ->", rejected["result"])


REJECTED -> Cancelled by human.


## 8. Approve / Reject / Modify Pattern

- **Approve:** `app.update_state(config, {"approved": True})` then `app.invoke(None, config)` — the graph finishes the run.
- **Reject:** `update_state` with `{"approved": False}` and resume so the cancel path runs.
- **Modify:** write the *edited* state fields (e.g. a corrected draft) via `update_state` so the next node sees the human's changes.

`app.get_state(config)` returns the snapshot at the interrupt, which you can log for an audit trail.

In [5]:
snap = app.get_state(config)
audit = {"thread": "approval-1", "planned_action": snap.values["action"],
         "decision": snap.values.get("approved")}
print("audit log entry:", audit)


audit log entry: {'thread': 'approval-1', 'planned_action': 'Send payment-reminder email to Ada', 'decision': True}



## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
